In [ ]:
# ============================================================
# STEP 1: INSTALL LIBRARIES
# ============================================================

!pip install transformers torch datasets -q

# ============================================================
# STEP 2: IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
# Should print: Using device: cuda

# ============================================================
# STEP 3: UPLOAD & LOAD DATA
# ===================================================

df = pd.read_csv('data.csv')  # ← change to your filename
df = df.dropna(subset=['question1', 'question2'])

# Sample 100k rows (Colab free GPU can handle this)
df = df.sample(n=99994, random_state=42).reset_index(drop=True)

print("✅ Dataset shape:", df.shape)
print(df['is_duplicate'].value_counts())

# ============================================================
# STEP 4: PREPARE DATA
# ============================================================

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['is_duplicate']
)

train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("✅ Train:", train_df.shape, "| Test:", test_df.shape)

# ============================================================
# STEP 5: TOKENIZER
# ============================================================

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
MAX_LEN   = 128  # max tokens per question pair

class QuoraDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df        = df
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        q1    = str(self.df['question1'][idx])
        q2    = str(self.df['question2'][idx])
        label = int(self.df['is_duplicate'][idx])

        # BERT takes two sentences as input separated by [SEP]
        encoding = self.tokenizer(
            q1, q2,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'token_type_ids': encoding['token_type_ids'].squeeze(),
            'label':          torch.tensor(label, dtype=torch.long)
        }

# Create datasets
train_dataset = QuoraDataset(train_df, tokenizer, MAX_LEN)
test_dataset  = QuoraDataset(test_df,  tokenizer, MAX_LEN)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print("✅ Data ready | Train batches:", len(train_loader), "| Test batches:", len(test_loader))

# ============================================================
# STEP 6: LOAD BERT MODEL
# ============================================================

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2           # duplicate or not duplicate
)
model = model.to(device)
print("✅ BERT model loaded")

# ============================================================
# STEP 7: OPTIMIZER & SCHEDULER
# ============================================================

EPOCHS = 3  # 3 epochs is enough for BERT

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

total_steps    = len(train_loader) * EPOCHS
scheduler      = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = total_steps // 10,
    num_training_steps = total_steps
)

print(f"✅ Training for {EPOCHS} epochs | {total_steps} total steps")

# ============================================================
# STEP 8: TRAINING LOOP
# ============================================================

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_preds  = []
    all_labels = []

    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )

        loss   = outputs.loss
        logits = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

        # Print progress every 100 batches
        if (batch_idx + 1) % 100 == 0:
            print(f"  Batch {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            token_type_ids = batch['token_type_ids'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels
            )

            loss   = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy, all_preds, all_labels


# ============================================================
# STEP 9: RUN TRAINING
# ============================================================

print("\n" + "="*50)
print("STARTING BERT TRAINING")
print("="*50)

best_accuracy = 0

for epoch in range(EPOCHS):
    print(f"\n📍 EPOCH {epoch+1}/{EPOCHS}")
    print("-"*40)

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f"✅ Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc*100:.2f}%")

    val_loss, val_acc, _, _ = evaluate(model, test_loader, device)
    print(f"✅ Val Loss:   {val_loss:.4f} | Val Accuracy:   {val_acc*100:.2f}%")

    # Save best model
    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), 'best_bert_model.pt')
        print(f"💾 Best model saved! Accuracy: {best_accuracy*100:.2f}%")

# ============================================================
# STEP 10: FINAL EVALUATION
# ============================================================

# Load best model
model.load_state_dict(torch.load('best_bert_model.pt'))

_, _, y_pred, y_true = evaluate(model, test_loader, device)

print("\n" + "="*50)
print("FINAL ACCURACY:", round(accuracy_score(y_true, y_pred) * 100, 2), "%")
print("="*50)
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Not Duplicate', 'Duplicate']))

✅ Using device: cuda
✅ Dataset shape: (99994, 28)
is_duplicate
0    63209
1    36785
Name: count, dtype: int64
✅ Train: (79995, 28) | Test: (19999, 28)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Data ready | Train batches: 2500 | Test batches: 625


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ BERT model loaded
✅ Training for 3 epochs | 7500 total steps

STARTING BERT TRAINING

📍 EPOCH 1/3
----------------------------------------
  Batch 100/2500 | Loss: 0.6943
  Batch 200/2500 | Loss: 0.5386
  Batch 300/2500 | Loss: 0.5437
  Batch 400/2500 | Loss: 0.4854
  Batch 500/2500 | Loss: 0.3111
  Batch 600/2500 | Loss: 0.3652
  Batch 700/2500 | Loss: 0.5310
  Batch 800/2500 | Loss: 0.3047
  Batch 900/2500 | Loss: 0.5048
  Batch 1000/2500 | Loss: 0.3273
  Batch 1100/2500 | Loss: 0.3938
  Batch 1200/2500 | Loss: 0.1937
  Batch 1300/2500 | Loss: 0.3743
  Batch 1400/2500 | Loss: 0.2351
  Batch 1500/2500 | Loss: 0.2987
  Batch 1600/2500 | Loss: 0.4525
  Batch 1700/2500 | Loss: 0.3933
  Batch 1800/2500 | Loss: 0.1797
  Batch 1900/2500 | Loss: 0.3195
  Batch 2000/2500 | Loss: 0.2965
  Batch 2100/2500 | Loss: 0.2108
  Batch 2200/2500 | Loss: 0.3811
  Batch 2300/2500 | Loss: 0.2054
  Batch 2400/2500 | Loss: 0.0627
  Batch 2500/2500 | Loss: 0.2640
✅ Train Loss: 0.3961 | Train Accuracy: 80.3

In [1]:
from transformers import BertTokenizer, BertForSequenceClassification

tokenizer = BertTokenizer.from_pretrained('rakhikumari/quora-duplicate-bert')
model     = BertForSequenceClassification.from_pretrained('rakhikumari/quora-duplicate-bert')

tokenizer.save_pretrained('../models/bert/')  # ← ../ because notebook is inside notebooks folder
model.save_pretrained('../models/bert/')
print("✅ BERT saved locally!")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ImportError: 
BertForSequenceClassification requires the PyTorch library but it was not found in your environment. Check out the instructions on the
installation page: https://pytorch.org/get-started/locally/ and follow the ones that match your environment.
Please note that you may need to restart your runtime after installation.
